# ELMo - Embeddings for Hotel Matching: Proof of Concept

In [ ]:
# System and logging
import os
from config import Config
import datetime as dt

# Database connection
import sqlalchemy as sqAl

# Used in cleaning rule
import re

# Data processing
import numpy as np
import pandas as pd

# Packages to use ELMo
import tensorflow as tf
import tensorflow_hub as hub

# Simple distance measure to identify match
from sklearn.metrics.pairwise import euclidean_distances as euc

# Jupyter notebook specific functions
from IPython.display import clear_output

## General Configuration

In [ ]:
# Load information for connection to Sybase
config_file: dict = Config(open("configs/login.cfg"))
assert set(["username", "pw", "host"]).issubset(config_file.keys()), "Config file not complete."

# Download pretrained ELMo-network
elmo = hub.Module("https://tfhub.dev/google/elmo/2", trainable=False)

In [ ]:
# Change values to adjust query for raw data
query_raw: dict   = {"year": 2018,
                     "month": 10,
                     "continent_id": 5}  # Africa: 1, Asia: 3, Australia: 4, Europe: 5, North America: 6, South America: 7

# Either choose number to draw a random sample or use "None" for complete data
city_sample: int = None

# Used to identify columns that need to be cleaned
cols_embeddings = ["HOTEL_NAME", "HOTEL_STREET"]

## Functions

In [ ]:
def read_pd_from_sybase(query: str, config: Config=config_file) -> pd.DataFrame:
        """
        Uses parameters from config in combination with sql query to load data from sybase to pd.DataFrame
        """
        engineString = f"sybase+pyodbc://{config.username}:{config.pw}@{config.host}"
        engine = sqAl.create_engine(engineString, echo=False)
        with engine.connect() as conn:
            return pd.read_sql(query, conn)

In [ ]:
def clean_with_simple_rule(s: pd.Series) -> pd.Series:
    """
    Returns strings in lower case with only alphanumeric characters
    """
    return s.map(str).apply(lambda r: re.sub("[^a-z0-9]", "", r.lower()))

In [ ]:
def return_ELMo_embeddings(strings: list) -> np.array:
    """
    Returns ELMo-embeddings for all elements of a list of strings
    """
    # Build minimalistic ELMo-network
    embedding_tensor = tf.unstack(elmo(strings, signature="default", as_dict=True)["elmo"])
    init_op = tf.global_variables_initializer()
    
    # Start tf Session and calculate embeddings
    with tf.Session() as sess:
        sess.run(init_op)
        calculated_embeddings = sess.run(embedding_tensor)
        
    return calculated_embeddings

In [ ]:
def calculate_ELMo_embeddings(feature: pd.Series) -> np.array:
    """
    Wrapper function to split long lists of strings into smaller chunks if necessary.
    """ 
    # Make sure that all elements have type str
    feature = [str(x) for x in feature]

    # Split feature into smaller chunks to avoid OOM error
    max_size = 4096
    if len(feature) > max_size:

        start = dt.datetime.now()

        index_steps = np.linspace(0, len(feature), num=int(np.ceil(len(feature)/max_size))+1, endpoint=True, dtype=int)  
        print(index_steps)
        elmo_list = list()

        for i in range(1, len(index_steps)):
            
            # Basic monitoring to estimate remaining time
            clear_output()
            print(f"Working on {i}/{len(index_steps)-1} slices.")
            if i > 1:
                print(f"Runtime:           {(dt.datetime.now()-start).seconds/60:2.2f} minutes.")
                print(f"Remaining runtime: {(((dt.datetime.now() - start) / (i-1)) * (len(index_steps)-(i-1))).seconds / 60:2.2f} minutes.")
                
            # Create next slice and calculate respective ELMo-embeddings
            next_slice = feature[index_steps[i-1]:index_steps[i]]
            embeddings = return_ELMo_embeddings(next_slice)
            elmo_list.append(np.reshape(np.array(embeddings), newshape=(len(next_slice), 1024)))

        # Combine embeddings of slices into one object
        calculated_elmos = np.concatenate(elmo_list, axis=0)

    else:
        embeddings = return_ELMo_embeddings(feature)
        calculated_elmos = np.reshape(np.array(embeddings), newshape=(len(feature), 1024))

    print(f"{dt.datetime.now()}: finished to calculate embeddings.")
    return calculated_elmos

In [ ]:
def return_indices_of_best_matches(raw_elmos: np.array, base_elmos: np.array) -> list:
    """
    Return indices based on distances between ELMo embeddings
    """
    distances = euc(raw_elmos, base_elmos)
    match_indices = list(np.argmin(distances, axis=1))
    return(match_indices)

## Load and prepare Raw Data

In [ ]:
%%time
# SQL query with placeholders that are filled according to configuration at the top of this notebook
sql_raw_data = f"SELECT a.HOTEL_NAME, a.HOTEL_STREET, b.HOTEL_CITY_ID as HOTEL_CITY_ID, b.HOTEL_NAME as MASTER_HOTELNAME " +\
               f"FROM DWHBIL.FAK_FOREIGN_DATA_NEW a " +\
               f"LEFT JOIN DWHBIL.V_LKP_HOTEL b on a.HOTEL_ID = b.HOTEL_ID " +\
               f"LEFT JOIN DWHBIL.V_LKP_HOTEL_CITY c on b.HOTEL_CITY_ID = c.HOTEL_CITY_ID " +\
               f"LEFT JOIN DWHBIL.V_LKP_CONTINENT d on b.HOTEL_CONTINENT_ID = d.CONTINENT_ID " +\
               f"WHERE year(a._LDTS) = {query_raw['year']} AND month(a._LDTS) = {query_raw['month']} AND ((a.HOTEL_NAME <>  b.HOTEL_NAME) OR (a.HOTEL_STREET <> b.HOTEL_STREET)) AND a.HOTEL_ID > 0 " +\
               f"AND b.HOTEL_CONTINENT_ID = {query_raw['continent_id']}"

# Collect raw data
raw_data = read_pd_from_sybase(sql_raw_data)
assert raw_data.shape[0] > 0, "No raw data fetched from database."
print(f"Shape of data fetched from Sybase: {raw_data.shape}")

# Draw subsample if city_sample is not None
relevant_cities = list(set(raw_data.HOTEL_CITY_ID))
print(f"Number of cities in raw data:\t{len(relevant_cities)}")
relevant_cities = relevant_cities if city_sample is None else list(np.random.choice(relevant_cities, city_sample))
raw_data = raw_data if city_sample is None else raw_data.loc[raw_data.HOTEL_CITY_ID.isin(relevant_cities)]
print(f"Number of sampled cities:\t{len(relevant_cities)}")

# Save original data for later comparision and reduce to unique values to increase efficiency
original_data = raw_data.copy()
print(f"Shape of raw data with duplicates:\t{raw_data.shape}")
raw_data.drop_duplicates(inplace=True)
print(f"Shape of unique records in raw data:\t{raw_data.shape}")
data = {"raw": raw_data, "original": original_data}

# Clean columns that are used for embeddings
for c in cols_embeddings:
    new_col = "clean_" + c
    data["raw"][new_col] = clean_with_simple_rule(data["raw"][c])
    
# Split data into records with and without street information
data["raw_name"] = data["raw"].loc[data["raw"].clean_HOTEL_STREET == ""].reset_index(drop=True)
data["raw_both"] = data["raw"].loc[data["raw"].clean_HOTEL_STREET != ""].reset_index(drop=True)
assert data["raw_name"].shape[0] + data["raw_both"].shape[0] == data["raw"].shape[0], "Unexpected value in 'clean_HOTEL_STREET'"
print(f"Number of unique rows with name only: \t\t{data['raw_name'].shape[0]}")
print(f"Number of unique rows with name and street:\t{data['raw_both'].shape[0]}")

In [ ]:
%%time
# Calculate ELMo-embeddings for the raw data
ELMo = {}
# Raw data might have complete information about street name
if data["raw_name"].shape[0] > 0:
    ELMo["raw_name"] = calculate_ELMo_embeddings(data["raw_name"].clean_HOTEL_NAME)
    
ELMo["raw_both"]  = np.concatenate([calculate_ELMo_embeddings(data["raw_both"].clean_HOTEL_NAME),
                                    calculate_ELMo_embeddings(data["raw_both"].clean_HOTEL_STREET)],
                                   axis=1)

## Collect Hotel Data and calculate ELMOs

In [ ]:
%%time
# Collect city IDs from raw data
list_cities = list(set(data["raw_name"].HOTEL_CITY_ID).union(set(data["raw_both"].HOTEL_CITY_ID)))
# Concatenate city IDs into the right format for SQL
string_cities = "(" + ",".join([str(x) for x in list_cities]) + ")"
# Create SQL query for hotel base
sql_master = f"SELECT b.HOTEL_NAME, b.HOTEL_STREET, c.HOTEL_CITY_ID " +\
                 f"FROM DWHBIL.V_LKP_HOTEL b " +\
                 f"JOIN DWHBIL.V_LKP_HOTEL_CITY c ON b.HOTEL_CITY_ID = c.HOTEL_CITY_ID " +\
                 f"WHERE b.HOTEL_CITY_ID in {string_cities}"
# Collect data from Sybase
hotel_base = read_pd_from_sybase(sql_master)
print(f"Number of hotels in the relevant set: {hotel_base.shape[0]}")

# Clean relevant columns for embeddings
for c in cols_embeddings:
    new_col = "clean_" + c
    hotel_base[new_col] = clean_with_simple_rule(hotel_base[c])

In [ ]:
%%time
# Calculate ELMO embeddings 
ELMo["base_name"] = calculate_ELMo_embeddings(hotel_base.clean_HOTEL_NAME)

ELMo["base_both"] = np.concatenate([ELMo["base_name"],
                                    calculate_ELMo_embeddings(hotel_base.clean_HOTEL_STREET)],
                                   axis=1)

## Loop over Cities to gather matches

In [ ]:
%%time

dfs_name = []
dfs_both = []

for city in relevant_cities:
    
    # Basic monitoring
    clear_output()
    print(f"Working on {relevant_cities.index(city) + 1}/{len(relevant_cities)} cities.")
    
    # Collect indices for the two types of raw data and the hotel base
    raw_name_indices = data["raw_name"].loc[data["raw_name"].HOTEL_CITY_ID == city].index
    raw_both_indices = data["raw_both"].loc[data["raw_both"].HOTEL_CITY_ID == city].index
    base_indices = list(hotel_base.loc[hotel_base.HOTEL_CITY_ID == city].index)
    
    # Make sure there is at least one record for that city in the data
    if len(raw_name_indices) > 0:

        match_indices = return_indices_of_best_matches(ELMo["raw_name"][raw_name_indices],
                                                       ELMo["base_name"][base_indices])
        hotel_matches = [hotel_base.HOTEL_NAME.iloc[base_indices[x]] for x in match_indices]
        
        df_name = data["raw_name"].iloc[raw_name_indices].copy()
        df_name["pred_match"] = hotel_matches
        dfs_name.append(df_name)
        
    # Make sure there is at least one record for that city in the data
    if len(raw_both_indices) > 0:
        
        match_indices = return_indices_of_best_matches(ELMo["raw_both"][raw_both_indices],
                                                       ELMo["base_both"][base_indices])
        hotel_matches = [hotel_base.HOTEL_NAME.iloc[base_indices[x]] for x in match_indices]
        
        df_both = data["raw_both"].iloc[raw_both_indices].copy()
        df_both["pred_match"] = hotel_matches
        dfs_both.append(df_both)
        
# Recombine raw data with and without street information
df_name = None if len(dfs_name) == 0 else pd.concat(dfs_name, axis=0, sort=False)
df_both = pd.concat(dfs_both, axis=0, sort=False)
df_matches = df_both if df_name is None else pd.concat([df_name, df_both], axis=0)
print(f"Correctly predicted {np.mean(df_matches.MASTER_HOTELNAME == df_matches.pred_match)*100:2.2f}% of unique entries.\n")

# Recombine matched from unique data with original dataset (including duplicates)
df_matches.set_index(["HOTEL_NAME", "HOTEL_STREET", "HOTEL_CITY_ID"], inplace=True)
df_matches.drop(["MASTER_HOTELNAME", "clean_HOTEL_NAME", "clean_HOTEL_STREET"], axis=1, inplace=True)
data["output"] = data["original"].join(df_matches, how="left", on=["HOTEL_NAME", "HOTEL_STREET", "HOTEL_CITY_ID"])
print(f"Overall accuracy on record level is \t{np.mean(data['output'].MASTER_HOTELNAME == data['output'].pred_match)*100:2.2f}% for {data['output'].shape[0]} records.")

### TIBO: notes

In [ ]:
# 11/2018
obs   = [4400, 231947, 2491, 26029, 106433, 2240]
match = [.9714, .9891, .9402, .9000, .9605, .9406]
print(f"Total accuracy of records: {np.sum([x[0] * x[1] for x in zip(obs, match)]) / np.sum(obs)*100:2.2f}% for {np.sum(obs)} observations.")

In [ ]:
# 12/2018
obs   = [60, 2837, 230, 6077, 1731, 58]
match = [.9, .9464, .9913, .9092, .9434, .9828]
print(f"Total accuracy of records: {np.sum([x[0] * x[1] for x in zip(obs, match)]) / np.sum(obs)*100:2.2f}% for {np.sum(obs)} observations.")